In [55]:
import pandas as pd
data = pd.read_csv("history_all_our.csv")

In [56]:
data.head()

,ir0,ir1,ir2,ir3,ir4,ir5,ir6,ir7,smell,move,turn
0,100,33,100,26,16,26,16,26.0,46.735705,6.431547,43.122617
1,28,100,33,20,31,20,30,100.0,3.644287,6.101230,0.607381
2,22,100,39,24,37,24,37,100.0,3.063364,7.285093,-5.489439
3,19,99,54,30,40,28,38,100.0,8.640332,6.549334,-11.903996
4,24,23,91,38,40,33,35,100.0,20.722835,6.975634,1.453806


In [57]:
data.describe()

,ir0,ir1,ir2,ir3,ir4,ir5,ir6,ir7,smell,move,turn
count,36489.000000,36489.000000,36489.000000,36489.000000,36489.000000,36489.000000,36489.000000,36489.000000,36489.000000,36489.000000,36488.000000
mean,75.456137,69.602949,60.495081,66.571131,82.374579,60.352188,54.155362,69.633381,-10.399932,5.521054,-1.475396
std,30.040557,32.559956,34.410590,32.513619,25.708973,32.176026,34.778443,31.820442,74.027440,2.230367,19.915911
min,0.000000,1.000000,0.000000,2.000000,1.000000,1.000000,0.000000,-3.576334,-179.937143,-35.929389,-110.000000
25%,50.000000,38.000000,26.000000,35.000000,67.000000,30.000000,21.000000,38.000000,-57.256174,4.230384,-10.589076
50%,96.000000,82.000000,56.000000,70.000000,100.000000,54.000000,41.000000,80.000000,-5.209599,5.625515,-0.666667
75%,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,35.357813,6.867097,8.855862
max,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,179.958820,15.930812,80.000000


In [58]:
#cleaning data
data = data[(data["move"] <= 10) & ( data["move"] >= -10)]

In [59]:
data.filter(items=['turn', 'move']).describe()

,turn,move
count,35737.000000,35737.000000
mean,-1.614854,5.409399
std,19.777384,2.095669
min,-110.000000,-1.176224
25%,-10.488843,4.195992
50%,-0.686182,5.573833
75%,8.461707,6.755166
max,80.000000,10.000000


In [60]:
def dist_to_label(dist: float) -> str:
    if dist < 10:
        return 'C'
    elif 10 <= dist < 25:
        return 'N'
    elif 25 <= dist < 50:
        return 'M'
    elif 50 <= dist < 75:
        return 'F'
    else:
        return 'L'

def angle_to_label(angle: float) -> str:
    if -10 <= angle <= 10:
        return 'C'
    elif 10 < angle < 45:
        return 'R'
    elif 45 <= angle < 90:
        return 'Z'
    elif 90 <= angle <= 180 or -180 <= angle <= -90:
        return 'B'
    elif -90 < angle <= -45:
        return 'A'
    elif -45 < angle < -10:
        return 'L'

In [61]:
data.iloc[:, 0:8] = data.iloc[:, 0:8].applymap(dist_to_label)
data.iloc[:, 8:9] = data.iloc[:, 8:9].applymap(angle_to_label)

C:\Users\bestk\AppData\Local\Temp\ipykernel_31828\4199426959.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  data.iloc[:, 0:8] = data.iloc[:, 0:8].applymap(dist_to_label)
C:\Users\bestk\AppData\Local\Temp\ipykernel_31828\4199426959.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0        L
1        M
2        N
3        N
4        N
        ..
36484    L
36485    L
36486    L
36487    L
36488    L
Name: ir0, Length: 35737, dtype: object' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.iloc[:, 0:8] = data.iloc[:, 0:8].applymap(dist_to_label)
C:\Users\bestk\AppData\Local\Temp\ipykernel_31828\4199426959.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0        M
1        L
2        L
3        L
4        N
        ..
36484    L
36485    F
36486    F
36487

In [62]:
data.iloc[:, 0:9].columns

Index(['ir0', 'ir1', 'ir2', 'ir3', 'ir4', 'ir5', 'ir6', 'ir7', 'smell'], dtype='object')

In [ ]:
def turnmove_to_class(row) -> str:
    turn = row['turn']
    move = row['move']
    action = '' 
    if -15 <= turn <= 15:
        action += 'F'
    if action != '':
        if turn < 0:
            turn+=360
        if turn > 180:
            action += 'L'
        else:
            action += 'R'
    if move < 0:
        action += '-1'
    elif move <= 5:
        action += '5'
    elif move > 5:
        action += '7'
    else:
        raise NotImplementedError()
    return action


In [64]:
data['action'] = data.apply(turnmove_to_class, axis=1)

In [ ]:
# Step 1: Initialize
pc = {}
px_given_c = {}


In [ ]:
# Step 2: Calculate Priors (The probability of just the Class)
C_count = data["action"].value_counts().to_dict()
for k in C_count:
    pc[k] = C_count[k] / len(data)

In [52]:
for col in data.iloc[:, 0:9].columns:
    print(f"Calculating P({col}|C)")
    px_given_c[col] = {}
    for possible_value in data[col].unique():
        px_given_c[col][possible_value] = {}
        for c in pc.keys():
            subset = data[data["action"] == c]
            value_count = len(subset[subset[col] == possible_value])
            total_count = len(subset)
            px_given_c[col][possible_value][c] = value_count / total_count

Calculating P(ir0|C)
Calculating P(ir1|C)
Calculating P(ir2|C)
Calculating P(ir3|C)
Calculating P(ir4|C)
Calculating P(ir5|C)
Calculating P(ir6|C)
Calculating P(ir7|C)
Calculating P(smell|C)


In [65]:
#inference 
data_test = data.head(1)

In [72]:
def inference(df: pd.DataFrame) -> str:
    row = df.iloc[0]
    scores = {}
    for c in pc.keys():
        score = pc[c]
        for col in df.iloc[:, 0:9].columns:
            value = row[col]
            score *= px_given_c[col][value][c]
        scores[c] = score
    predicted_class = max(scores, key=scores.get)
    return predicted_class

In [73]:
inference(data_test)

'R5'